# **서로소 집합(Disjoint Set, Union-Find)**

그래프나 집합 문제를 풀 때, **몇 개의 원소들이 서로 연결되어 있는지**, **서로 다른 그룹(집합)으로 분리되어 있는지**를 빠르게 관리하고 싶을 때가 있다.

이를 효율적으로 처리하는 자료구조가 바로 **서로소 집합(Disjoint Set)**, 또는 유니온 파인드(Union-Find)라 불린다.

서로소 집합(Disjoint Set)은 원소들의 집합이 여러 그룹으로 나뉘어 있고, 그룹들 사이에 겹치는 원소가 없다는(서로소) 특징을 효율적으로 관리하기 위한 자료구조다.

## **1. 개념**

1. **서로소 집합(Disjoint Set)**
    - 서로소 또는 상호배타 집합들은 서로 중복 포함된 원소가 없는 집합들이다. 다시 말해 교집합이 없다.
    - 두 집합 간에 공통 원소가 하나도 없을 때, 이 두 집합을 서로소 집합이라고 한다.
        - 서로 다른 그룹들 간에는 **겹치는 원소**가 없음 → "서로소"
    - 원소들이 여러 그룹으로 나뉘어 있을 때, 두 원소가 **동일한 그룹**에 속하는지, **다른 그룹**인지 판단
2. **Union-Find**
    - 보통 **Union** 연산(합치기)과 **Find** 연산(루트 노드를 찾기, 같은 집합인지 확인)을 제공
    - 그래프에서의 **연결성** 체크
        - 예: “두 노드가 같은 컴포넌트인가?”나, **사이클** 검사, **최소 신장 트리(MST)** 등에서 활약

### **1.1 왜 필요한가?**

- 예: 네트워크에서 컴퓨터 간 연결(같은 네트워크 그룹인지), 또는 그래프에서 사이클이 생기는지 검사
- 만약 두 원소가 **같은 그룹**에 속하는지, **다른 그룹**인지 빠르게 판별하고 싶음
- 한 번 두 그룹이 연결되었으면(같은 집합이 되었으면), 계속해서 그 정보를 갱신하면서 관리

### **1.2 주요 연산**

원소 x가 속한 **집합**을 대표하는 **루트(root)**를 찾아냄

- '대표' 혹은 '부모' 개념을 이용: x가 속한 최상위 부모 노드를 찾아서 반환
1. **Union(a, b)**
    - a가 속한 집합과 b가 속한 집합을 **합침**(하나의 그룹으로)
    - 즉, a의 루트와 b의 루트를 찾아 “둘 다 다르면” 한쪽 루트를 다른 쪽의 부모로 만든다

## **2. 단계별 예시**

### **2.1 단계 [1]: 가장 단순한 Union-Find**

1. **초기 상태**
    - 각 원소가 자기 자신을 부모로 함
    - `parent[i] = i`
2. **Union(a, b)**
    - a의 루트를 find, b의 루트를 find
    - 만약 다르면 b의 루트를 a의 루트로 바꾸거나, 그 반대
3. **Find(x)**
    - x가 속한 루트를 찾기 위해, parent[x]를 따라가며 **루트**(parent== x)가 될 때까지 이동

**예시**

- 집합에 1,2,3,4,5가 있다고 가정
- 초기: `parent[1]=1`, `parent[2]=2` ...
- Union(1,2) → parent[2] = 1 (또는 parent[1] = 2)
- Find(2) → 결과: 1 (2의 루트는 1)

### 2.2 단계 [2]: 코드 (간단 버전, 경로 압축 없음)

In [ ]:
def find_parent(parent_list, x):
    # 부모가 자기 자신일 때까지 따라간다
    while parent_list[x] != x:
        x = parent_list[x]
    return x

def union_parent(parent_list, a, b):
    root_a = find_parent(parent_list, a)
    root_b = find_parent(parent_list, b)
    if root_a != root_b:
        # 간단히 a 루트를 b 루트에 연결
        parent_list[root_b] = root_a

# 사용 예시
n = 5
parent = [i for i in range(n + 1)]  # 1~5

union_parent(parent, 1, 2)
union_parent(parent, 2, 3)

print(find_parent(parent, 3))  # 1 (root)

**해설**

- `find_parent`: 단순 while문으로 위로 올라감
- `union_parent`: 두 루트 다르면 한쪽을 다른 쪽 부모로
- 이런 방식은 **경로 압축 없이** 구현, 크기가 커질수록 루트를 찾는 시간이 오래 걸릴 수 있음

### **2.3 단계 [3]: 경로 압축 & Union By Rank**

- **경로 압축**
    - find 시, 루트를 찾으면 **중간 노드**들의 부모도 바로 루트로 설정 → 나중에 더 빠르게 접근
- **Union By Rank**
    - 높은 랭크(트리 높이) 혹은 더 큰 집합 쪽이 부모가 되도록 합침 → 트리 깊이 증가 방지

In [ ]:
def find_parent(parent_list, x):
    """
    find_parent: x의 루트(대표자)를 찾아 반환.
    경로 압축(Path Compression) 기법 사용.
    parent_list[x] = x 인 경우, x가 루트.
    """
    if parent_list[x] != x:
        # '경로 압축': 재귀 종료 후 부모를 루트로 바로 설정
        parent_list[x] = find_parent(parent_list, parent_list[x])
    return parent_list[x]

def union_parent(parent_list, rank_list, a, b):
    """
    union_parent: 두 원소 a, b를 같은 집합으로 합침 (Union by Rank).
    - 먼저 a와 b의 루트를 찾고,
    - rank(또는 높이)가 낮은 쪽을 높은 쪽 밑에 붙임.
    """
    root_a = find_parent(parent_list, a)
    root_b = find_parent(parent_list, b)

    # 루트가 같으면 이미 같은 집합
    if root_a == root_b:
        return

    # rank(높이) 비교해서 더 낮은 쪽을 높은 쪽 밑으로 붙임
    if rank_list[root_a] < rank_list[root_b]:
        parent_list[root_a] = root_b
    elif rank_list[root_a] > rank_list[root_b]:
        parent_list[root_b] = root_a
    else:
        # rank가 같으면 한 쪽을 다른 쪽 아래로 붙이고 rank+1
        parent_list[root_b] = root_a
        rank_list[root_a] += 1

# 예: 1부터 N까지 노드가 있다고 가정
n = 5
parent = [i for i in range(n + 1)]  # parent[i] = i 초기화
rank = [0] * (n + 1)  # 높이(또는 랭크)

# union 연산 수행
union_parent(parent, rank, 1, 2)
union_parent(parent, rank, 2, 3)
union_parent(parent, rank, 4, 5)

# find 연산 테스트
print(find_parent(parent, 3))  # 예: 1
print(find_parent(parent, 4))  # 예: 4 or 5

# 두 노드가 같은 집합인지 확인
def same_set(a, b):
    return find_parent(parent, a) == find_parent(parent, b)

print("1과 3은 같은 집합?", same_set(1, 3))  # True
print("2와 5는 같은 집합?", same_set(2, 5))  # False

**코드 설명**

1. **parent_list**: 각 노드의 부모(대표). `parent_list= x`이면, x는 루트(자기 자신이 대표)
2. **find_parent**
    - x의 루트를 찾는다. 찾는 과정에서 **경로 압축**을 적용 → 한번 찾으면 부모를 루트로 업데이트
3. **union_parent**
    - a, b 각각의 루트를 찾고, 다르면 합침
    - **union by rank**로, 낮은 rank를 높은 rank 쪽에 붙인다
4. **예시**
    - 1,2,3은 서로 union → 같은 그룹
    - 4,5은 서로 union → 같은 그룹
    - `same_set(a,b)`는 `find_parent(a) == find_parent(b)`인지로 판별

### **2.4 유망한 기법들**

- **경로 압축(Path Compression)**
    - `Find` 과정에서, 찾은 루트로 **재빠르게 연결**해주어 탐색 시간을 단축
    - 만나는 모든 노드들이 직접 root를 가리키도록 포인터를 바꾸어 줌
    
- **랭크/크기 기반 합치기(Union by rank/size)**
    - 항상 **더 작은 트리**를 **더 큰 트리** 밑으로 붙임 → 트리 깊이를 줄여 효율성 향상

## **3. 응용 예시**

1. **사이클 검사**
    - 무방향 그래프에서 간선을 순서대로 확인하며, 두 노드가 이미 **같은 집합**이면 사이클 발생
2. **최소 스패닝 트리(크루스칼 알고리즘)**
    - 간선들을 가중치 오름차순 정렬 → 순서대로 간선을 확인하며, 두 노드가 **다른 집합**이면 union → MST 완성
3. **네트워크 연결**
    - 컴퓨터(노드)들 중에서 **연결된 그룹**을 유지
    - 두 컴퓨터 연결 시 union, 연결 여부 질의 시 find로 **동일 루트** 여부 확인
4. **소셜 미디어** 등, 유저가 속한 그룹을 추적

## **4. 시간 복잡도**

- 경로 압축과 랭크(또는 사이즈) 최적화를 사용하면, find/union 연산이 **거의 `$O(1)$`**(정확히 `$O(α(N)$`) 이지만 `$α(N)$`은 매우 느리게 증가)
- 대규모 그래프(수십만 노드)에서도 **효율적**으로 동작

## **5. 정리**

<aside>
💡

“**같은 집합이면 루트가 같다**”는 원리와 “**Union-Find**로 합치고 찾는다”는 개념을 이해하기

</aside>

1. **서로소 집합(Disjoint Set)** 또는 **Union-Find**: 여러 원소를 그룹으로 묶고, 그 그룹이 어떤 집합인지 빠르게 판별하고 합치는 자료구조
2. **핵심 연산**
    - **Find**(어떤 원소 x의 루트를 찾는)
    - **Union**(두 그룹을 합치는)
3. **최적화 기법**: 경로 압축(Path Compression), 랭크 기반 합치기(Union by Rank)
4. **활용**: 사이클 검사, MST(Kruskal), 네트워크 연결성, 소셜미디어 그룹 등등
5. 이후 **최소 신장 트리**나 **최단 경로** 알고리즘에서 그래프 간선 연결 여부를 처리할 때 자주 만나게 됨